# Trabajo Practico 4 - Medidas Electrónicas 1
**Docentes:**

*   Ing. Marinsek, Emiliano
*   Ing. Perdomo, Juan Manuel

**Alumnos:**

*   Ferrario, Franco Ezequiel
*   Encinas, Leandro
*   Marchesi, Matias
*   Quiroga, Bruno

**Curso:** R4052

**Año:** 2026


# Librerias y funciones a utilizar

In [1]:
# Librerias de pyhon a utilizar
import numpy as np
import math
import IPython
from IPython.display import display, Math
import pandas as pd
from typing import Any


def incert_A(mediciones):
  s = np.std(mediciones, ddof=1, )  # ddof=1 -> grados de libertad v = 1/n-1
  return s/math.sqrt(len(mediciones))

def incert_B(err_rel, nd, ctas, mediciones):
  return (err_rel/100 + nd/ctas) * np.mean(mediciones) / math.sqrt(3)


def incert_B_ConMedia(err_rel, nd, ctas, media):
  return (err_rel/100 + nd/ctas) * media / math.sqrt(3)

def incert_Total(uA, uB):
  return math.sqrt(uA**2 + uB**2)

def incert_Comb_Rel(uX, uY,correlacion=0):
  return math.sqrt((uX['u_C']/uX['media'])**2 + (uY['u_C']/uY['media'])**2 - (2*(uX['u_C']/uX['media'])*(uY['u_C']/uY['media'])*correlacion))


def incert_B_DSO(V, V_div):
  # Exactitud vertical del DSO: ±(3% lectura + 0.1 div + 1 mV), distribución rectangular
  return (0.03*V + 0.1*V_div + 0.001) / math.sqrt(3)

def propagar(func, x, u):
  # GUM: u_c^2 = sum (df/dx_i * u_i)^2, con derivadas parciales numéricas
  suma = 0
  for i in range(len(x)):
    h = u[i] * 1e-3
    xp = list(x); xm = list(x)
    xp[i] += h; xm[i] -= h
    c_i = (func(*xp) - func(*xm)) / (2*h)
    suma += (c_i * u[i])**2
  return math.sqrt(suma)

def formato(valor, U, unidad, k=2):
  # U con 2 cifras significativas y valor acotado a esos dígitos
  dec = 1 - int(math.floor(math.log10(abs(U))))
  U_r = round(U, dec)
  if U_r >= 10**(2 - dec):  # el redondeo agregó una cifra (p. ej. 9.96 -> 10)
    dec -= 1
    U_r = round(U, dec)
  d = max(dec, 0)
  return f"({valor:.{d}f} ± {U_r:.{d}f}) {unidad} [k={k}]"

# Ejercicio 1
Medición de impedancia.

Se busca medir la impedancia de un dipolo paralelo conformado por un  capacitor y una resistencia cuya fase se encuentra entre 25º/40º a una frecuencia de 1kHz. Para ello se propone medir la tensión y corriente del dipolo utilizando un DSO y un resistencia en serie para medir la corriente.
Nota: Tenga cuidado de definir las resistencias para que las mediciones de  tensión esté en la misma escala y de no cortocircuitar las masas del DSO.


Cálculo teórico (Se realizó con f=10kHz):

$Z_P$ = $\frac{R_2}{1+- j \omega C R_2 }$

$Z_{Total}$ = $R_1$ + $\frac{R_2}{1+ j \omega C R_2 }$

$Z_{Total}$ = ($R_1$ + $\frac{R_2}{1+ (\omega C R_2)^2 }$) - j ($\frac{\omega C R_2^2}{1+ (\omega C R_2)^2 }$)

$H$ = $\frac{V_{Out}}{V_{In}}$ = $\frac{Z_{P}}{Z_{Total}}$
$H$ = $\frac{247.05 - j 1552.23}{1247.05 -j1552.23}$

$\theta_{num}$ = arctan($\frac{-1552.23}{247.05}$) = $-80.96º$
$\theta_{den}$ = arctan($\frac{-1552.23}{1247.05}$) = $-51.22º$

$\phi$ = $\theta_{num}$ - $\theta_{den}$ = $-80.96º$  $+51.22º$ = $-29.74º$

$\Delta{t}$ = $\frac{|\phi|}{360º * f}$ =$8.26\mu s$

# Medición y cálculos del laboratorio

<img src="recursos/ej1_circuito.png" width="400">

*Circuito de medición: CH1 en Ch_A (generador) y CH2 en Ch_B (dipolo). La corriente se obtiene con MATH CH1−CH2 sobre R1.*

<img src="recursos/ej1_desfasaje_dipolo.jpeg" width="600">

*Desfasaje entre la tensión del generador (CH1) y la del dipolo (CH2) medido con cursores.*

<img src="recursos/ej1_tension_R_math.jpeg" width="600">

*MATH CH1−CH2 (rojo): tensión sobre la resistencia de medición, proporcional a la corriente del dipolo.*

In [2]:
delta_t = 8e-6 #s
esc_hor = 50e-6  #s/div
res_cursores = 0.1e-6 #s Resolucion del cursor

u_base_tiempo = (delta_t * 0.01)/(100*math.sqrt(3))

u_res_dig = math.sqrt(((res_cursores/2)/math.sqrt(3))**2+((res_cursores/2)/math.sqrt(3))**2)

u_medicion_visual = (res_cursores)/math.sqrt(3)

u_c = math.sqrt(u_base_tiempo**2 + u_res_dig**2 + u_medicion_visual**2)
print("u(delta_t): "+ str(u_c))

print("U(delta_t): "+ str(u_c*2)) #k=2

print(f"Δt = ({delta_t*10**6:.2f} ± {(u_c*2)*10**6:.2f}) μs [k=2]")



u(delta_t): 7.071218659703102e-08
U(delta_t): 1.4142437319406204e-07
Δt = (8.00 ± 0.14) μs [k=2]


## Medición de la impedancia

Con CH1 en la salida del generador (Ch_A) y CH2 sobre el dipolo (Ch_B), la tensión sobre $R_m$ se obtiene con la función MATH (CH1 − CH2), por lo que la corriente es $I = V_R/R_m$ sin cortocircuitar las masas del DSO. Se midieron con cursores las tensiones pico $V_{dip}$ y $V_R$ y el desfasaje $\Delta t$ entre $V_{gen}$ y $V_{dip}$. $R_m$ se midió 10 veces con el tester UNI-T UT33C.

Módulo:

$$|Z| = \frac{V_{dip}}{I} = \frac{V_{dip}}{V_R}\,R_m$$

Fase: como $\vec V_{gen} = \vec V_{dip} + \vec V_R$ y la corriente está en fase con $V_R$, del triángulo fasorial (teorema del seno):

$$\alpha = 360º\, f\, \Delta t \qquad \gamma = \arcsin\left(\frac{V_{dip}\,\sin\alpha}{V_R}\right) \qquad \varphi_Z = -(\alpha + \gamma)$$

Componentes equivalentes del paralelo: $Y = \frac{1}{Z} = \frac{1}{R_p} + j\,\omega\, C_p$

Incertidumbre combinada según la GUM: $u_c^2(y) = \sum_i \left(\frac{\partial y}{\partial x_i}\right)^2 u^2(x_i)$

In [3]:
k = 2
f = 9.9992e3   # Hz (frecuencímetro del DSO)
w = 2*math.pi*f
V_div = 0.5    # V/div (CH2 y MATH)

# Resistencia de medición R_m: tester UNI-T UT33C, rango 2 kΩ: ±(0.8% + 2 d)
R_m_med = [1002, 1001, 1002, 1001, 1001, 1001, 1001, 1001, 1001, 1002]
R_m = np.mean(R_m_med)
u_R_m = incert_Total(incert_A(R_m_med), incert_B_ConMedia(0.8, 2, 1001, R_m))

# Tensiones pico medidas con cursores
V_dip = 0.700  # V, CH2 (dipolo)
V_R = 0.480    # V, MATH CH1-CH2 (sobre R_m)
u_V_dip = incert_B_DSO(V_dip, V_div)
u_V_R = incert_B_DSO(V_R, V_div)

# Desfasaje V_gen -> V_dip (celda anterior)
dt = delta_t
u_dt = u_c

def Z_mod(V_dip, V_R, R_m, dt):
  return V_dip / V_R * R_m

def Z_fase(V_dip, V_R, R_m, dt):
  alfa = w * dt
  gamma = math.asin(V_dip * math.sin(alfa) / V_R)
  return -math.degrees(alfa + gamma)

def R_p(*x):
  return 1 / (1 / (Z_mod(*x) * np.exp(1j*math.radians(Z_fase(*x))))).real

def C_p(*x):
  return (1 / (Z_mod(*x) * np.exp(1j*math.radians(Z_fase(*x))))).imag / w

x = [V_dip, V_R, R_m, dt]
u = [u_V_dip, u_V_R, u_R_m, u_dt]

Z = Z_mod(*x)
phi = Z_fase(*x)
U_Z = k * propagar(Z_mod, x, u)
U_phi = k * propagar(Z_fase, x, u)
U_Rp = k * propagar(R_p, x, u)
U_Cp = k * propagar(C_p, x, u)

# Valor teórico con los componentes medidos
R2 = 9.91e3
C = 10e-9
Z_teo = R2 / (1 + 1j*w*C*R2)

print(f"R_m = {formato(R_m, k*u_R_m, 'Ω')}")
print(f"|Z| = {formato(Z/1e3, U_Z/1e3, 'kΩ')}")
print(f"φ_Z = {formato(phi, U_phi, '°')}")
print(f"R_p = {formato(R_p(*x)/1e3, U_Rp/1e3, 'kΩ')}")
print(f"C_p = {formato(C_p(*x)*1e9, U_Cp*1e9, 'nF')}")
print(f"Teórico: |Z| = {abs(Z_teo):.0f} Ω, φ_Z = {math.degrees(np.angle(Z_teo)):.1f}°")

R_m = (1001 ± 12) Ω [k=2]
|Z| = (1.46 ± 0.29) kΩ [k=2]
φ_Z = (-73 ± 11) ° [k=2]
R_p = (5.1 ± 4.4) kΩ [k=2]
C_p = (10.4 ± 1.5) nF [k=2]
Teórico: |Z| = 1572 Ω, φ_Z = -80.9°


# Conclusion

La impedancia medida del dipolo a $f \approx 10\,kHz$ es $|Z| = (1.46 \pm 0.29)\,k\Omega$ con $\varphi_Z = (-73 \pm 11)º$ [k=2], frente al valor teórico de $1.57\,k\Omega \angle -80.9º$ calculado con $R = 9.91\,k\Omega$ y $C = 10\,nF$; el valor teórico queda dentro del intervalo de incertidumbre.

La capacidad equivalente obtenida, $C_p = (10.4 \pm 1.5)\,nF$, coincide con la nominal. En cambio $R_p$ queda muy mal determinada, porque con una fase cercana a $-90º$ la parte real de la admitancia es chica y muy sensible al error de fase.

La incertidumbre está dominada por la exactitud vertical del DSO (3% + 0.1 div), ya que las tensiones ocupan pocas divisiones; usar una escala más sensible la reduciría.

# Ejercicio 2
Mediciones de eventos únicos.

Utilizando un circuito conformado por un relé, un switch y un diodo removible se busca medir:


*   Tiempo de apertura y tiempo de cierre (con y sin diodo de protección)
*   Sobre-tensión en la llave en el momento de apertura (con y sin diodo de protección)




**Tiempo de establecimiento**

<img src="recursos/ej2_tiempo_establecimiento.jpeg" width="600">

*Tiempo de establecimiento de la llave.*

**Tiempo de establecimiento CON diodo**

<img src="recursos/ej2_con_diodo.jpeg" width="600">

*Con diodo de protección.*

**Tiempo de establecimiento SIN diodo**

<img src="recursos/ej2_sin_diodo.jpeg" width="600">

*Sin diodo de protección.*

Sin diodo podemos observar un sobrepico de 18.7 $V_{PP}$ y un tiempo de establecimiento de 590 $ns$
Con diodo se puede observar un menor sobrepico, de 12.6 $V_{PP}$, pero el tiempo de establecimiento es de 560 $\mu s$


# Calculos de incertidumbre CON DIODO

In [4]:
import numpy as np
# INCERTIDUMBRE DEL TIEMPO DE ESTABLECIMIENTO
print("TIEMPO DE ESTABLECIMIENTO:")

# Datos iniciales
t_s = 560e-6  # 560 us en segundos
exactitud_horizontal = 0.01 / 100  # 0.01%

#incertidumbre (distribución rectangular)
delta_t = t_s * exactitud_horizontal
u_t_s = delta_t / np.sqrt(3)

# incertidumbre expandida (k=2)
k = 2
U_t_s = k * u_t_s

print(f"Valor observado (ts): {t_s * 1e6:.2f} us")
print(f"Incertidumbre estándar u(ts): {u_t_s * 1e9:.2f} ns")
print(f"Incertidumbre expandida U(ts) [k=2]: {U_t_s * 1e9:.2f} ns")
print(f"Resultado final: ({t_s * 1e6:.3f} ± {U_t_s * 1e6:.3f}) us\n")


# ==========================================
#INCERTIDUMBRE DEL SOBREPICO
# ==========================================
print("SOBREPICO:")

# Datos de tensión para el Canal 1
V_max = 12.6  # Voltios
V_div = 5.0   # Voltios por división (Escala vertical)
V_estable = 12.0  # Voltios (Valor final de régimen estable estimado)

# Especificaciones del Datasheet para Exactitud Vertical
# ±(3% * Lectura + 0.1 div + 1 mV)
error_lectura = 0.03 * V_max
error_division = 0.1 * V_div
error_fijo = 0.001

delta_V = error_lectura + error_division + error_fijo
u_V_max = delta_V / np.sqrt(3)

# Cálculo del sobrepico porcentual
overshoot = ((V_max - V_estable) / V_estable) * 100

# Propagación de incertidumbre según la GUM (Derivada parcial d(Overshoot)/dV_max)
# Coeficiente de sensibilidad c = 100 / V_estable
c_sensibilidad = 100 / V_estable
u_overshoot = c_sensibilidad * u_V_max

# Incertidumbre expandida (k=2)
U_overshoot = k * u_overshoot

print(f"Valor calculado de Overshoot: {overshoot:.2f}%")
print(f"Incertidumbre estándar u(Overshoot): {u_overshoot:.2f}%")
print(f"Incertidumbre expandida U(Overshoot) [k=2]: {U_overshoot:.2f}%")
print(f"Resultado final: Overshoot = ({overshoot:.1f} ± {U_overshoot:.1f})%")

TIEMPO DE ESTABLECIMIENTO:
Valor observado (ts): 560.00 us
Incertidumbre estándar u(ts): 32.33 ns
Incertidumbre expandida U(ts) [k=2]: 64.66 ns
Resultado final: (560.000 ± 0.065) us

SOBREPICO:
Valor calculado de Overshoot: 5.00%
Incertidumbre estándar u(Overshoot): 4.23%
Incertidumbre expandida U(Overshoot) [k=2]: 8.46%
Resultado final: Overshoot = (5.0 ± 8.5)%


# Calculos de Incertidumbre SIN DIODO

In [5]:
import numpy as np
# INCERTIDUMBRE DEL TIEMPO DE ESTABLECIMIENTO
print("TIEMPO DE ESTABLECIMIENTO:")

t_s = 590e-9  # 590 ns expresado en segundos
exactitud_horizontal = 0.01 / 100  # 0.01% según el datasheet

# Cálculo bajo distribución rectangular (GUM Tipo B)
delta_t = t_s * exactitud_horizontal
u_t_s = delta_t / np.sqrt(3)
k = 2
U_t_s = k * u_t_s

print(f"Valor observado (ts): {t_s * 1e9:.3f} ns")
print(f"Incertidumbre estándar u(ts): {u_t_s * 1e9:.3f} ns")
print(f"Incertidumbre expandida U(ts) [k=2]: {U_t_s * 1e9:.3f} ns")
print(f"Resultado final: ts = ({t_s * 1e9:.3f} ± {U_t_s * 1e9:.3f}) ns \n")


# ==========================================================
# INCERTIDUMBRE DEL SOBREPICO
# ==========================================================
print(" SOBREPICO:")

# Datos obtenidos de la pantalla para el Canal 1
V_max = 18.7      # Valor Vmax medido en pantalla (Voltios)
V_estable = 12.0  # Nivel de régimen permanente (3 divisiones x 5V/div = 12V)
V_div = 5.0       # Escala vertical (Voltios/división)

# Tolerancia vertical del instrumento: ±(3%*Lectura + 0.1 div + 1 mV)
error_lectura = 0.03 * V_max
error_division = 0.1 * V_div
error_fijo = 0.001

delta_V = error_lectura + error_division + error_fijo
u_V_max = delta_V / np.sqrt(3)

# Cálculo del sobrepico porcentual relativo al nivel estable
overshoot_porcentaje = ((V_max - V_estable) / V_estable) * 100

# Derivada parcial de Overshoot respecto a Vmax: c = 100 / V_estable
c_sensibilidad = 100 / V_estable
u_overshoot = c_sensibilidad * u_V_max
U_overshoot = k * u_overshoot

print(f"Valor Vmax: {V_max:.1f} V")
print(f"Incertidumbre estándar u(Vmax): {u_V_max:.4f} V")
print(f"Sobrepico calculado: {overshoot_porcentaje:.2f}%")
print(f"Incertidumbre estándar u(Overshoot): {u_overshoot:.2f}%")
print(f"Incertidumbre expandida U(Overshoot) [k=2]: {U_overshoot:.2f}%")
print(f"Resultado final: Overshoot = ({overshoot_porcentaje:.0f} ± {U_overshoot:.0f})%")

TIEMPO DE ESTABLECIMIENTO:
Valor observado (ts): 590.000 ns
Incertidumbre estándar u(ts): 0.034 ns
Incertidumbre expandida U(ts) [k=2]: 0.068 ns
Resultado final: ts = (590.000 ± 0.068) ns 

 SOBREPICO:
Valor Vmax: 18.7 V
Incertidumbre estándar u(Vmax): 0.6131 V
Sobrepico calculado: 55.83%
Incertidumbre estándar u(Overshoot): 5.11%
Incertidumbre expandida U(Overshoot) [k=2]: 10.22%
Resultado final: Overshoot = (56 ± 10)%


# Conclusiones
Podemos observar claramente que el no tener diodo en el circuito, nos da un sobrepico mayor, pero el tiempo de establecimiento es menor  gracias al sobrepico producido por la inducción de la bobina del relé. En contraparte, vemos que el circuito con diodo, si bien se retrasa más, es más seguro eléctricamente hablando, si la velocidad no es una prioridad para el circuito, ya que el sobrepico es menor pero el tiempo de establecimiento es bastante mayor.

# Ejercicio 3
Tiempo de establecimiento

Medir el tiempo de establecimiento de un generador de señal rectangular.

<img src="recursos/ej3_rise_time_generador.jpeg" width="600">

*Tiempo de subida (10% - 90%) del generador medido con cursores.*

**Aporte del sistema de medición**

El tiempo de subida medido incluye el del osciloscopio y el de la punta:

$$t_{r,med} = \sqrt{t_{r,gen}^2 + t_{r,DSO}^2 + t_{r,punta}^2} \quad\Rightarrow\quad t_{r,gen} = \sqrt{t_{r,med}^2 - t_{r,DSO}^2 - t_{r,punta}^2}$$

con $t_r \approx 0.35/BW$:
* GDS-1102A-U: $BW = 100\,MHz \Rightarrow t_{r,DSO} \approx 3.5\,ns$
* Punta en x10: $BW = 100\,MHz \Rightarrow t_{r,punta} \approx 3.5\,ns$ (en x1 el ancho de banda típico es de unos 6 MHz, $t_r \approx 58\,ns$, mayor que lo medido, por lo que la medición se hizo en x10).

Para $t_{r,DSO}$ y $t_{r,punta}$ se toma una incertidumbre rectangular de ±20% por ser valores aproximados.

In [6]:
k = 2
t_med = 49.00e-9      # s, cursores X1X2 al 10% y 90%
res_cursores = 1e-9   # s, resolución del cursor a 25 ns/div

u_base_tiempo = t_med * 0.01/100 / math.sqrt(3)                  # exactitud base de tiempo 0.01%
u_res_dig = math.sqrt(2) * (res_cursores/2) / math.sqrt(3)       # resolución de los dos cursores
u_medicion_visual = res_cursores / math.sqrt(3)                  # ubicación visual de los cursores
u_t_med = math.sqrt(u_base_tiempo**2 + u_res_dig**2 + u_medicion_visual**2)

t_DSO = 0.35 / 100e6
t_punta = 0.35 / 100e6
u_t_DSO = 0.2 * t_DSO / math.sqrt(3)
u_t_punta = 0.2 * t_punta / math.sqrt(3)

def t_gen(t_med, t_DSO, t_punta):
  return math.sqrt(t_med**2 - t_DSO**2 - t_punta**2)

x = [t_med, t_DSO, t_punta]
u = [u_t_med, u_t_DSO, u_t_punta]
t_r_gen = t_gen(*x)
U_t_r_gen = k * propagar(t_gen, x, u)

print(f"t_r medido      = {formato(t_med*1e9, k*u_t_med*1e9, 'ns')}")
print(f"t_r DSO + punta = {math.sqrt(t_DSO**2 + t_punta**2)*1e9:.1f} ns")
print(f"t_r generador   = {formato(t_r_gen*1e9, U_t_r_gen*1e9, 'ns')}")

t_r medido      = (49.0 ± 1.4) ns [k=2]
t_r DSO + punta = 4.9 ns
t_r generador   = (48.7 ± 1.4) ns [k=2]


# Conclusiones

Con los cursores ubicados al 10% y al 90% de la señal se midió $t_{r,med} = (49.0 \pm 1.4)\,ns$ [k=2]. Descontando el aporte del osciloscopio y de la punta en x10 (≈ 4.9 ns combinados), el tiempo de subida del generador es $t_{r,gen} = (48.7 \pm 1.4)\,ns$ [k=2].

La corrección es chica (≈ 0.3 ns) porque el sistema de medición es unas 10 veces más rápido que la señal; con la punta en x1 la medición no hubiese sido posible. Para esta medición se conectó el osciloscopio directamente al generador de señal.

# Ejercicio 4
Compensación de la punta

A partir de una señal senoidal de 100 kHz y Vpp = 4 V, medir el máximo error de tensión que se puede producir utilizando una punta sin compensar frente a otra compensada.  


Imagen del osciloscopio:

<img src="recursos/ej4_compensacion_punta.jpeg" width="600">

*CH1: punta sin compensar, CH2: punta compensada.*

In [7]:
# Factor de cobertura
k = 2

print("COMPARATIVA DE PUNTAS DE PRUEBA:")

# Datos de la pantalla
Vpp_sin_compensar = 5.03  # CH1
Vpp_compensada = 4.00     # CH2
V_div = 2.0               # Escala vertical de 2 V/div para ambos

# Error medido directamente
error_medido = Vpp_sin_compensar - Vpp_compensada

# uB_CH1
delta_V1 = (0.03 * Vpp_sin_compensar) + (0.1 * V_div) + 0.001
u_V1 = delta_V1 / np.sqrt(3)

# uB_CH2
delta_V2 = (0.03 * Vpp_compensada) + (0.1 * V_div) + 0.001
u_V2 = delta_V2 / np.sqrt(3)

# Propagación de incertidumbre para la diferencia (Error = V1 - V2)
u_error = np.sqrt(u_V1**2 + u_V2**2)
U_error = k * u_error

# Máximo error posible (Límite superior del intervalo)
max_error_posible = error_medido + U_error

print(f"Error de tensión observado: {error_medido:.2f} V")
print(f"Incertidumbre expandida de la diferencia (k=2): ±{U_error:.2f} V")
print(f"Intervalo del error: ({error_medido:.2f} ± {U_error:.2f}) V")
print(f"MÁXIMO ERROR REAL QUE SE PUEDE PRODUCIR: {max_error_posible:.2f} V")

COMPARATIVA DE PUNTAS DE PRUEBA:
Error de tensión observado: 1.03 V
Incertidumbre expandida de la diferencia (k=2): ±0.55 V
Intervalo del error: (1.03 ± 0.55) V
MÁXIMO ERROR REAL QUE SE PUEDE PRODUCIR: 1.58 V


# Conclusiones
La falta de compensación en la punta de prueba del Canal 1 introdujo un error sistemático severo en la amplitud, registrando un valor de 5.03 $V_{pp}$ frente a los 4.00 $V_{pp}$ de la punta calibrada (un error medido de 1.03 $V$). Al propagar las incertidumbres instrumentales, se determinó que la incertidumbre de la diferencia es de $0.55V_{pp}$, lo que sitúa al máximo error de tensión posible en $1.58V_{pp}$.
Esto nos hace notar que debemos compensar bien las puntas antes de cualquier medicion ya que sino distorsiona por completo el valor de la medicion.